# Разведка `dfs_5min.pickle`

Цель ноутбука — понять устройство файла с котировками фьючерсов перед реализацией стратегий и двухслойного фильтра.

Ноутбук ничего не торгует и не меняет исходный файл. Он выводит мета-информацию, типы объектов, ключи, размеры таблиц, индексы, колонки, пропуски, временное покрытие и первые строки выборочных таблиц.

Важная временная дисциплина для будущего `backtest (исторического тестирования)`: признаки в момент `t` должны использовать только данные, доступные до решения в `t`, а `PnL (прибыль и убыток)` считается на позиции, удерживаемой до следующего движения цены.

In [43]:
from pathlib import Path
import json
import os
import pickle
import sys
from collections import Counter

import numpy as np
import pandas as pd

In [44]:
DATA_PATH = Path('dfs_5min.pickle')

if not DATA_PATH.exists():
    raise FileNotFoundError(f'Файл не найден рядом с ноутбуком: {DATA_PATH.resolve()}')

stat = DATA_PATH.stat()
file_meta = pd.DataFrame([{
    'path': str(DATA_PATH.resolve()),
    'size_bytes': stat.st_size,
    'size_gb': stat.st_size / 1024**3,
    'modified_utc': pd.to_datetime(stat.st_mtime, unit='s', utc=True),
}])
file_meta

,path,size_bytes,size_gb,modified_utc
0,C:\Users\jack\Jupyter_files\QUANT_TRAIDING\Fit...,1458525408,1.358358,2026-09-12 09:26:11.031317234+00:00


## Загрузка объекта

Файл большой, поэтому эта ячейка может занять время и память. После загрузки дальнейшие ячейки работают с уже загруженным объектом.

In [45]:
with DATA_PATH.open('rb') as f:
    dfs = pickle.load(f)

type(dfs), sys.getsizeof(dfs)

(dict, 832)

In [46]:
def short_repr(value, max_len=160):
    text = repr(value)
    return text if len(text) <= max_len else text[:max_len] + '...'


def object_summary(obj, name='dfs'):
    rows = [{
        'name': name,
        'type': type(obj).__name__,
        'len': len(obj) if hasattr(obj, '__len__') else np.nan,
        'repr': short_repr(obj),
    }]
    if isinstance(obj, dict):
        rows[0]['keys_sample'] = list(obj.keys())[:10]
    if isinstance(obj, pd.DataFrame):
        rows[0]['shape'] = obj.shape
        rows[0]['index_type'] = type(obj.index).__name__
        rows[0]['columns_sample'] = list(obj.columns[:12])
    return pd.DataFrame(rows)


object_summary(dfs)

,name,type,len,repr,keys_sample
0,dfs,dict,35,{'ZC': Date Symbol Op...,"[ZC, ZS, ZL, ZM, ZW, KE, ZO, ZR, RS, CL]"


## Верхний уровень структуры

In [47]:
if isinstance(dfs, dict):
    top_rows = []
    for key, value in dfs.items():
        row = {
            'key': key,
            'value_type': type(value).__name__,
            'len': len(value) if hasattr(value, '__len__') else np.nan,
            'repr': short_repr(value, 120),
        }
        if isinstance(value, pd.DataFrame):
            row.update({
                'shape': value.shape,
                'index_type': type(value.index).__name__,
                'columns': list(value.columns),
            })
        elif isinstance(value, dict):
            row['nested_keys_sample'] = list(value.keys())[:12]
        top_rows.append(row)
    top_summary = pd.DataFrame(top_rows)
else:
    top_summary = object_summary(dfs)

top_summary

,key,value_type,len,repr,shape,index_type,columns
0,ZC,DataFrame,744746,Date Symbol Open H...,"(744746, 9)",RangeIndex,"[Date, Symbol, Open, High, Low, Close, Volume,..."
1,ZS,DataFrame,764096,Date Symbol Open ...,"(764096, 9)",RangeIndex,"[Date, Symbol, Open, High, Low, Close, Volume,..."
2,ZL,DataFrame,735005,Date Symbol Open Hig...,"(735005, 9)",RangeIndex,"[Date, Symbol, Open, High, Low, Close, Volume,..."
3,ZM,DataFrame,703413,Date Symbol Open Hig...,"(703413, 9)",RangeIndex,"[Date, Symbol, Open, High, Low, Close, Volume,..."
4,ZW,DataFrame,724802,Date Symbol Open H...,"(724802, 9)",RangeIndex,"[Date, Symbol, Open, High, Low, Close, Volume,..."
5,KE,DataFrame,508681,Date Symbol Open H...,"(508681, 9)",RangeIndex,"[Date, Symbol, Open, High, Low, Close, Volume,..."
6,ZO,DataFrame,213905,Date Symbol Open H...,"(213905, 9)",RangeIndex,"[Date, Symbol, Open, High, Low, Close, Volume,..."
7,ZR,DataFrame,197996,Date Symbol Open H...,"(197996, 9)",RangeIndex,"[Date, Symbol, Open, High, Low, Close, Volume,..."
8,RS,DataFrame,350416,Date Symbol Open Hig...,"(350416, 9)",RangeIndex,"[Date, Symbol, Open, High, Low, Close, Volume,..."
9,CL,DataFrame,1048570,Date Symbol Open Hi...,"(1048570, 9)",RangeIndex,"[Date, Symbol, Open, High, Low, Close, Volume,..."


In [48]:
def iter_dataframes(obj, path='dfs', max_depth=4):
    if max_depth < 0:
        return
    if isinstance(obj, pd.DataFrame):
        yield path, obj
    elif isinstance(obj, dict):
        for key, value in obj.items():
            yield from iter_dataframes(value, f'{path}[{key!r}]', max_depth - 1)
    elif isinstance(obj, (list, tuple)):
        for idx, value in enumerate(obj):
            yield from iter_dataframes(value, f'{path}[{idx}]', max_depth - 1)


frames = list(iter_dataframes(dfs))
len(frames)

35

## Сводка по таблицам

In [49]:
frame_rows = []
for path, frame in frames:
    idx = frame.index
    row = {
        'path': path,
        'rows': len(frame),
        'cols': frame.shape[1],
        'index_type': type(idx).__name__,
        'index_name': idx.name,
        'is_monotonic_increasing': bool(idx.is_monotonic_increasing),
        'duplicate_index_count': int(idx.duplicated().sum()) if hasattr(idx, 'duplicated') else np.nan,
        'memory_mb': frame.memory_usage(deep=True).sum() / 1024**2,
        'columns': list(frame.columns),
    }
    if isinstance(idx, pd.DatetimeIndex):
        row['start'] = idx.min()
        row['end'] = idx.max()
        row['tz'] = str(idx.tz)
        if len(idx) > 2:
            diffs = idx.to_series().diff().dropna()
            row['median_step'] = diffs.median()
            row['mode_step'] = diffs.mode().iloc[0] if not diffs.mode().empty else pd.NaT
    frame_rows.append(row)

frames_summary = pd.DataFrame(frame_rows).sort_values(['rows', 'cols'], ascending=False)
frames_summary

,path,rows,cols,index_type,index_name,is_monotonic_increasing,duplicate_index_count,memory_mb,columns
9,dfs['CL'],1048570,9,RangeIndex,None,True,0,117.999451,"[Date, Symbol, Open, High, Low, Close, Volume,..."
25,dfs['E6'],1048570,9,RangeIndex,None,True,0,117.999451,"[Date, Symbol, Open, High, Low, Close, Volume,..."
27,dfs['B6'],1048570,9,RangeIndex,None,True,0,117.999451,"[Date, Symbol, Open, High, Low, Close, Volume,..."
29,dfs['A6'],1048570,9,RangeIndex,None,True,0,117.999451,"[Date, Symbol, Open, High, Low, Close, Volume,..."
30,dfs['J6'],1048570,9,RangeIndex,None,True,0,117.999451,"[Date, Symbol, Open, High, Low, Close, Volume,..."
26,dfs['D6'],1043303,9,RangeIndex,None,True,0,117.406736,"[Date, Symbol, Open, High, Low, Close, Volume,..."
31,dfs['S6'],1020486,9,RangeIndex,None,True,0,114.839058,"[Date, Symbol, Open, High, Low, Close, Volume,..."
12,dfs['NG'],959491,9,RangeIndex,None,True,0,107.975073,"[Date, Symbol, Open, High, Low, Close, Volume,..."
32,dfs['N6'],957654,9,RangeIndex,None,True,0,107.768349,"[Date, Symbol, Open, High, Low, Close, Volume,..."
28,dfs['DX'],935991,9,RangeIndex,None,True,0,105.330534,"[Date, Symbol, Open, High, Low, Close, Volume,..."


In [50]:
column_counter = Counter()
dtype_rows = []
for path, frame in frames:
    column_counter.update(map(str, frame.columns))
    for col, dtype in frame.dtypes.items():
        dtype_rows.append({'path': path, 'column': col, 'dtype': str(dtype)})

pd.DataFrame(column_counter.most_common(), columns=['column', 'frame_count'])

,column,frame_count
0,Date,35
1,Symbol,35
2,Open,35
3,High,35
4,Low,35
5,Close,35
6,Volume,35
7,OpenInterest,35
8,SaleCondition,35


In [51]:
pd.DataFrame(dtype_rows).pivot_table(index='column', columns='dtype', values='path', aggfunc='count', fill_value=0)

dtype,datetime64[ns],float64,int64,object
column,,,,
Close,0,33,2,0
Date,35,0,0,0
High,0,33,2,0
Low,0,33,2,0
Open,0,33,2,0
OpenInterest,0,35,0,0
SaleCondition,0,35,0,0
Symbol,0,0,0,35
Volume,0,0,35,0


## Пропуски и базовая проверка качества

In [52]:
quality_rows = []
for path, frame in frames:
    total_cells = frame.shape[0] * frame.shape[1]
    na_cells = int(frame.isna().sum().sum())
    numeric = frame.select_dtypes(include='number')
    quality_rows.append({
        'path': path,
        'rows': frame.shape[0],
        'cols': frame.shape[1],
        'na_cells': na_cells,
        'na_pct': na_cells / total_cells if total_cells else np.nan,
        'numeric_cols': numeric.shape[1],
        'inf_count': int(np.isinf(numeric.to_numpy()).sum()) if numeric.shape[1] else 0,
    })

pd.DataFrame(quality_rows).sort_values('na_pct', ascending=False)

,path,rows,cols,na_cells,na_pct,numeric_cols,inf_count
0,dfs['ZC'],744746,9,1489492,0.222222,7,0
1,dfs['ZS'],764096,9,1528192,0.222222,7,0
2,dfs['ZL'],735005,9,1470010,0.222222,7,0
3,dfs['ZM'],703413,9,1406826,0.222222,7,0
4,dfs['ZW'],724802,9,1449604,0.222222,7,0
5,dfs['KE'],508681,9,1017362,0.222222,7,0
6,dfs['ZO'],213905,9,427810,0.222222,7,0
7,dfs['ZR'],197996,9,395992,0.222222,7,0
8,dfs['RS'],350416,9,700832,0.222222,7,0
9,dfs['CL'],1048570,9,2097140,0.222222,7,0


In [53]:
for path, frame in frames[:10]:
    print('\n' + '=' * 100)
    print(path)
    display(frame.head(3))
    display(frame.tail(3))


dfs['ZC']


,Date,Symbol,Open,High,Low,Close,Volume,OpenInterest,SaleCondition
0,2023-08-10 11:00:00,ZCZ23,496.75,497.00,496.25,496.25,278,NaN,NaN
1,2023-08-10 10:55:00,ZCZ23,497.25,497.50,496.75,496.75,1040,NaN,NaN
2,2023-08-10 10:50:00,ZCZ23,497.25,497.25,496.50,497.25,538,NaN,NaN


,Date,Symbol,Open,High,Low,Close,Volume,OpenInterest,SaleCondition
744743,2008-05-04 18:10:00,ZCN08,613.50,613.75,612.75,613.00,1,NaN,NaN
744744,2008-05-04 18:05:00,ZCN08,615.00,615.25,613.00,613.50,5,NaN,NaN
744745,2008-05-04 18:00:00,ZCN08,616.25,616.50,610.50,614.75,28,NaN,NaN



dfs['ZS']


,Date,Symbol,Open,High,Low,Close,Volume,OpenInterest,SaleCondition
0,2023-08-10 11:00:00,ZSX23,1321.25,1321.5,1319.75,1319.75,382,NaN,NaN
1,2023-08-10 10:55:00,ZSX23,1320.75,1322.5,1320.75,1321.50,269,NaN,NaN
2,2023-08-10 10:50:00,ZSX23,1322.00,1322.0,1319.75,1320.50,446,NaN,NaN


,Date,Symbol,Open,High,Low,Close,Volume,OpenInterest,SaleCondition
764093,2008-05-04 18:30:00,ZSN08,1295.50,1295.50,1292.50,1292.5,1,NaN,NaN
764094,2008-05-04 18:10:00,ZSN08,1294.00,1295.75,1293.25,1295.0,1,NaN,NaN
764095,2008-05-04 18:00:00,ZSN08,1305.25,1306.75,1296.50,1299.0,3,NaN,NaN


... (20 outputs left out)

,Date,Symbol,Open,High,Low,Close,Volume,OpenInterest,SaleCondition
350413,2008-05-05 13:00:00,RSN08,599.2,599.6,596.3,599.6,1,NaN,NaN
350414,2008-05-05 12:45:00,RSN08,596.0,598.0,595.0,598.0,1,NaN,NaN
350415,2008-05-05 09:30:00,RSN08,608.0,611.1,608.0,609.8,3,NaN,NaN



dfs['CL']


,Date,Symbol,Open,High,Low,Close,Volume,OpenInterest,SaleCondition
0,2023-08-10 11:00:00,CLU23,83.60,83.71,83.50,83.69,1251,NaN,NaN
1,2023-08-10 10:55:00,CLU23,83.67,83.72,83.57,83.60,987,NaN,NaN
2,2023-08-10 10:50:00,CLU23,83.69,83.72,83.58,83.66,1347,NaN,NaN


,Date,Symbol,Open,High,Low,Close,Volume,OpenInterest,SaleCondition
1048567,2008-11-06 04:55:00,CLZ08,64.49,64.57,64.46,64.46,81,NaN,NaN
1048568,2008-11-06 04:50:00,CLZ08,64.63,64.63,64.42,64.45,69,NaN,NaN
1048569,2008-11-06 04:45:00,CLZ08,64.38,64.72,64.36,64.58,62,NaN,NaN


## Поиск ценовых колонок для будущих стратегий

In [54]:
PRICE_HINTS = ('close', 'open', 'high', 'low', 'bid', 'ask', 'mid', 'price', 'settle', 'last', 'base')

price_rows = []
for path, frame in frames:
    for col in frame.columns:
        col_text = str(col).lower()
        if any(hint in col_text for hint in PRICE_HINTS):
            series = frame[col]
            numeric = pd.to_numeric(series, errors='coerce')
            price_rows.append({
                'path': path,
                'column': col,
                'dtype': str(series.dtype),
                'non_na': int(series.notna().sum()),
                'min': numeric.min(),
                'max': numeric.max(),
                'mean': numeric.mean(),
                'std': numeric.std(),
            })

price_candidates = pd.DataFrame(price_rows)
price_candidates

,path,column,dtype,non_na,min,max,mean,std
0,dfs['ZC'],Open,float64,744746,302.000000,839.750000,485.792858,140.563339
1,dfs['ZC'],High,float64,744746,302.250000,849.000000,486.099789,140.666534
2,dfs['ZC'],Low,float64,744746,302.000000,838.750000,485.492909,140.457553
3,dfs['ZC'],Close,float64,744746,302.000000,839.750000,485.791326,140.564396
4,dfs['ZC'],OpenInterest,float64,0,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...
170,dfs['T6'],Open,float64,188411,0.050175,0.151575,0.067199,0.010931
171,dfs['T6'],High,float64,188411,0.050200,0.151575,0.067210,0.010930
172,dfs['T6'],Low,float64,188411,0.050150,0.151575,0.067189,0.010932
173,dfs['T6'],Close,float64,188411,0.050200,0.151575,0.067199,0.010931


## Временное покрытие и регулярность 5-минутных данных

In [55]:
time_rows = []
for path, frame in frames:
    if isinstance(frame.index, pd.DatetimeIndex):
        idx = frame.index.sort_values()
        diffs = idx.to_series().diff().dropna()
        time_rows.append({
            'path': path,
            'start': idx.min(),
            'end': idx.max(),
            'rows': len(idx),
            'median_step': diffs.median() if len(diffs) else pd.NaT,
            'share_5min_steps': float((diffs == pd.Timedelta(minutes=5)).mean()) if len(diffs) else np.nan,
            'largest_gap': diffs.max() if len(diffs) else pd.NaT,
        })

time_coverage = pd.DataFrame(time_rows)
if not time_coverage.empty and 'start' in time_coverage.columns:
    time_coverage = time_coverage.sort_values('start')
time_coverage


""


## Сохранение результатов аудита

Этот блок сохраняет сводные таблицы в папку `audit_dfs_5min`. Папка создаётся рядом с ноутбуком. Исходный `dfs_5min.pickle` не изменяется.


In [56]:
AUDIT_DIR = Path('audit_dfs_5min')
AUDIT_DIR.mkdir(exist_ok=True)
AUDIT_DIR.resolve()


WindowsPath('C:/Users/jack/Jupyter_files/QUANT_TRAIDING/Fiter_2options/audit_dfs_5min')

In [57]:
# Сохраняем основные таблицы аудита.
file_meta.to_csv(AUDIT_DIR / '00_file_meta.csv', index=False, encoding='utf-8-sig')
top_summary.to_csv(AUDIT_DIR / '01_top_level_structure.csv', index=False, encoding='utf-8-sig')
frames_summary.to_csv(AUDIT_DIR / '02_frames_summary.csv', index=False, encoding='utf-8-sig')
pd.DataFrame(dtype_rows).to_csv(AUDIT_DIR / '03_column_dtypes_long.csv', index=False, encoding='utf-8-sig')
pd.DataFrame(column_counter.most_common(), columns=['column', 'frame_count']).to_csv(
    AUDIT_DIR / '04_column_frequency.csv', index=False, encoding='utf-8-sig'
)
pd.DataFrame(quality_rows).sort_values('na_pct', ascending=False).to_csv(
    AUDIT_DIR / '05_quality_summary.csv', index=False, encoding='utf-8-sig'
)
if 'price_candidates' in globals() and isinstance(price_candidates, pd.DataFrame):
    price_candidates.to_csv(AUDIT_DIR / '06_price_column_candidates.csv', index=False, encoding='utf-8-sig')
if 'time_coverage' not in globals():
    time_coverage = pd.DataFrame(time_rows) if 'time_rows' in globals() else pd.DataFrame()
if not time_coverage.empty and 'start' in time_coverage.columns:
    time_coverage = time_coverage.sort_values('start')
time_coverage.to_csv(AUDIT_DIR / '07_time_coverage.csv', index=False, encoding='utf-8-sig')

saved_files = sorted(path.name for path in AUDIT_DIR.glob('*.csv'))
pd.DataFrame({'saved_file': saved_files})


,saved_file
0,00_file_meta.csv
1,01_top_level_structure.csv
2,02_frames_summary.csv
3,03_column_dtypes_long.csv
4,04_column_frequency.csv
5,05_quality_summary.csv
6,06_price_column_candidates.csv
7,07_time_coverage.csv


In [58]:
# Компактный JSON-манифест: удобно приложить к отчёту или быстро прочитать без открытия всех CSV.
audit_manifest = {
    'source_file': str(DATA_PATH.resolve()),
    'source_size_bytes': int(stat.st_size),
    'audit_dir': str(AUDIT_DIR.resolve()),
    'top_object_type': type(dfs).__name__,
    'top_len': int(len(dfs)) if hasattr(dfs, '__len__') else None,
    'dataframe_count': int(len(frames)),
    'total_rows_across_frames': int(sum(len(frame) for _, frame in frames)),
    'total_memory_mb_across_frames': float(sum(frame.memory_usage(deep=True).sum() for _, frame in frames) / 1024**2),
    'saved_csv_files': saved_files,
}

with (AUDIT_DIR / '08_audit_manifest.json').open('w', encoding='utf-8') as f:
    json.dump(audit_manifest, f, ensure_ascii=False, indent=2, default=str)

audit_manifest


{'source_file': 'C:\\Users\\jack\\Jupyter_files\\QUANT_TRAIDING\\Fiter_2options\\dfs_5min.pickle',
 'source_size_bytes': 1458525408,
 'audit_dir': 'C:\\Users\\jack\\Jupyter_files\\QUANT_TRAIDING\\Fiter_2options\\audit_dfs_5min',
 'top_object_type': 'dict',
 'top_len': 35,
 'dataframe_count': 35,
 'total_rows_across_frames': 21169205,
 'total_memory_mb_across_frames': 2382.2506046295166,
 'saved_csv_files': ['00_file_meta.csv',
  '01_top_level_structure.csv',
  '02_frames_summary.csv',
  '03_column_dtypes_long.csv',
  '04_column_frequency.csv',
  '05_quality_summary.csv',
  '06_price_column_candidates.csv',
  '07_time_coverage.csv']}

In [59]:
readme = f'''# Аудит dfs_5min.pickle

Папка создана ноутбуком `explore_dfs_5min_pickle.ipynb`.

Файлы:

- `00_file_meta.csv` — путь, размер и дата изменения исходного файла.
- `01_top_level_structure.csv` — верхний уровень объекта после загрузки `pickle`.
- `02_frames_summary.csv` — размеры, индексы, колонки и память найденных таблиц.
- `03_column_dtypes_long.csv` — типы колонок в длинном формате.
- `04_column_frequency.csv` — как часто встречается каждая колонка.
- `05_quality_summary.csv` — пропуски, числовые колонки и бесконечные значения.
- `06_price_column_candidates.csv` — вероятные ценовые колонки для будущих стратегий.
- `07_time_coverage.csv` — временное покрытие и регулярность 5-минутного шага.
- `08_audit_manifest.json` — компактный машинночитаемый манифест аудита.

Исходный файл не изменяется.
'''

(AUDIT_DIR / 'README.md').write_text(readme, encoding='utf-8')
print((AUDIT_DIR / 'README.md').resolve())


./data\audit_dfs_5min\README.md


## Минимальный вывод для следующего шага

После запуска ноутбука стоит зафиксировать:

1. какие ключи верхнего уровня соответствуют инструментам;
2. где лежат торгуемые цены для стратегий;
3. есть ли отдельные `bid (лучшая цена покупателя)` и `ask (лучшая цена продавца)` или только `close (цена закрытия)`;
4. как устроены `base_1`, `base_2`, `base_3`, если они есть;
5. насколько регулярны 5-минутные временные метки;
6. какие данные можно использовать для сигналов, а какие только для диагностики.